In [1]:
#load MOAChi modules
%run MO_Atm_coupling_101325_test.ipynb


In [2]:
# read the inputs CSV as a pandas DataFrame
static_model_outputs_df = pd.read_csv('static_model_outputs_csv.csv')

# quick check
print("Loaded static_model_outputs_csv.csv:", static_model_outputs_df.shape)
static_model_outputs_df.head()

Loaded static_model_outputs_csv.csv: (29, 32)


,planet,τ,Case,Model,t(yr),T_surf(K),T_pot(K),flux_surf(W/m2),flux_OLR(W/m2),flux_ASR(W/m2),...,p_H2O(bar),p_CO2(bar),p_CO(bar),p_H2(bar),p_CH4(bar),p_O2(bar),mmw(kg/mol),R_trans(m),R_solid(m),viscosity(Pa.s)
0,trappist-1b,3,Hot,neongooey,1000293.29,3411.364461,3440.133685,26600.596344,199087.582531,172484.536217,...,13.682752,148.857499,NaN,9.998345258055E+00,NaN,1.709426758034E-03,0.039515,NaN,3.712860e+06,4.790759152231E-07
1,trappist-1b,3,Cold,pacman,1002.80,2835.298084,2909.511630,84384.287315,92548.186063,8163.674068,...,2.884216,79.275577,56.09503107251123,0.20938287213330373,1.8266453242449353e-10,0.08723831092948514,0.037139,NaN,4.440306e+06,0.0026694609635622398
2,trappist-1b,4,Hot,neongooey,9800.85,3363.801397,3363.914852,16.241328,172500.746349,172484.536217,...,14.644827,150.564062,NaN,9.983981560214E+00,NaN,1.775308272419E-03,0.039444,NaN,3.889201e+06,5.090982524255E-07
3,trappist-1b,4,Cold,pacman,10011.80,2245.948182,2255.186010,3450.757844,11613.461365,8162.621180,...,8.908084,117.177692,30.96467939928224,0.35703719404638,1.933527078532965e-09,0.0011607366280315951,0.039505,NaN,5.720448e+06,0.009371555690607096
4,trappist-1b,6,Hot,neongooey,1016695.54,3276.413204,3276.485200,8.638637,136712.919094,136704.274456,...,15.873727,151.770303,NaN,8.471530718298E+00,NaN,1.865828348110E-03,0.039647,NaN,4.082946e+06,5.486031498306E-07


In [3]:
#convert columns to float, except for the first, third and fourth columns (strings)
cols_keep = static_model_outputs_df.columns[[0, 2, 3]]
cols_convert = static_model_outputs_df.columns.difference(cols_keep)

# replace literal "NaN" strings, then convert selected columns to float
static_model_outputs_df[cols_convert] = (
    static_model_outputs_df[cols_convert]
    .replace("NaN", 0)
    .apply(pd.to_numeric, errors="coerce")
    .fillna(0.0)
    .astype(float)
)

static_model_outputs_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 29 entries, 0 to 28
Data columns (total 32 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   planet             29 non-null     object 
 1   τ                  29 non-null     float64
 2   Case               29 non-null     object 
 3   Model              29 non-null     object 
 4   t(yr)              29 non-null     float64
 5   T_surf(K)          29 non-null     float64
 6   T_pot(K)           29 non-null     float64
 7   flux_surf(W/m2)    29 non-null     float64
 8   flux_OLR(W/m2)     29 non-null     float64
 9   flux_ASR(W/m2)     29 non-null     float64
 10  phi(vol_frac)      29 non-null     float64
 11  fO2_solid(bar)     29 non-null     float64
 12  fO2_melt(bar)      29 non-null     float64
 13  thick_surf_bl(m)   29 non-null     float64
 14  massC_solid(kg)    29 non-null     float64
 15  massC_melt(kg)     29 non-null     float64
 16  massC_atm(kg)      29 non-nu

In [ ]:
#total mass of C, N, H, O in the atmosphere (kg)
m_CNHO_atms = np.array([static_model_outputs_df['massC_atm(kg) '].values,
                        np.zeros_like(static_model_outputs_df['massC_atm(kg) '].values), 
                        static_model_outputs_df['massH_atm(kg) '].values,
                        static_model_outputs_df['massO_atm(kg) '].values])

In [8]:
m_CNHO_atms = m_CNHO_atms.T  # Transpose to shape (N, 4) for MOAChi input

In [ ]:
# Normalize the mass fractions to sum to 1
m_CNHO_atms = m_CNHO_atms/np.sum(m_CNHO_atms, axis=1)[:, np.newaxis] 

In [5]:
print('Stephan-Boltzmann constant (W/m2/K4):', sigma_SB)
#calculate the internal temperature from the surface flux using the Stefan-Boltzmann law
T_ints = (static_model_outputs_df['flux_surf(W/m2) ']/sigma_SB)**0.25

Stephan-Boltzmann constant (W/m2/K4): 5.670374419e-08


In [6]:
#set the mass and radius arrays to be used in the MOAChi model
#other than the last 12 to 6 entries, use 1 Earth mass and radius.
#The last 12 to 6 entries are Venus values.
Ms = np.ones_like(T_ints)
Rs = np.ones_like(T_ints)
Ms[-12:-6] = 0.815
Rs[-12:-6] = 0.95

In [7]:
#calculate the equilibrium temperature from the absorbed stellar flux using the Stefan-Boltzmann law
T_eqs = (static_model_outputs_df['flux_ASR(W/m2) ']/sigma_SB)**0.25

In [9]:
#extract the surface pressure from the DataFrame and convert from bar to Pa
P_surfs = static_model_outputs_df['p_surf(bar) '].values*1e5  # convert bar to Pa

In [ ]:
# Extract and reorder gas partial pressures: O2, H2, CO2, CO, CH4, H2O. 
#MOAChi also needs N2 which is 0 for all cases in this dataset.
p_gases_ordered_df = pd.DataFrame({
    'O2':  static_model_outputs_df['p_O2(bar) '],
    'H2':  static_model_outputs_df['p_H2(bar) '],
    'CO2': static_model_outputs_df['p_CO2(bar) '],
    'CO':  static_model_outputs_df['p_CO(bar) '],
    'CH4': static_model_outputs_df['p_CH4(bar) '],
    'H2O': static_model_outputs_df['p_H2O(bar) '],
    'N2':  np.zeros(len(static_model_outputs_df), dtype=float)
})

# Optional ndarray form (shape: n_rows x 7)
p_gases_ordered = p_gases_ordered_df.to_numpy()

p_gases_ordered_df.head()

,O2,H2,CO2,CO,CH4,H2O,N2
0,0.001709,9.998345,148.857499,0.000000,0.000000e+00,13.682752,0.0
1,0.087238,0.209383,79.275577,56.095031,1.826645e-10,2.884216,0.0
2,0.001775,9.983982,150.564062,0.000000,0.000000e+00,14.644827,0.0
3,0.001161,0.357037,117.177692,30.964679,1.933527e-09,8.908084,0.0
4,0.001866,8.471531,151.770303,0.000000,0.000000e+00,15.873727,0.0


In [11]:
# calculate CNHO mass fractions from atmospheric partial pressures (H2O, CO2, CO, H2, CH4, O2)

p_H2O = static_model_outputs_df['p_H2O(bar) '].to_numpy(float)
p_CO2 = static_model_outputs_df['p_CO2(bar) '].to_numpy(float)
p_CO  = static_model_outputs_df['p_CO(bar) '].to_numpy(float)
p_H2  = static_model_outputs_df['p_H2(bar) '].to_numpy(float)
p_CH4 = static_model_outputs_df['p_CH4(bar) '].to_numpy(float)
p_O2  = static_model_outputs_df['p_O2(bar) '].to_numpy(float)

# Species "moles" are proportional to partial pressures (ideal gas mixture, same T/V)
n_H2O = p_H2O
n_CO2 = p_CO2
n_CO  = p_CO
n_H2  = p_H2
n_CH4 = p_CH4
n_O2  = p_O2

# Element moles (N not present in these 6 gases)
n_C = n_CO2 + n_CO + n_CH4
n_N = np.zeros_like(n_C)
n_H = 2*n_H2O + 2*n_H2 + 4*n_CH4
n_O = n_H2O + 2*n_CO2 + n_CO + 2*n_O2

# Convert to element masses (kg per mol)
m_C = n_C * 0.012
m_N = n_N * 0.014
m_H = n_H * 0.001
m_O = n_O * 0.016

m_tot = m_C + m_N + m_H + m_O
valid = m_tot > 0

w_C = np.zeros_like(m_tot)
w_N = np.zeros_like(m_tot)
w_H = np.zeros_like(m_tot)
w_O = np.zeros_like(m_tot)

w_C[valid] = m_C[valid] / m_tot[valid]
w_N[valid] = m_N[valid] / m_tot[valid]
w_H[valid] = m_H[valid] / m_tot[valid]
w_O[valid] = m_O[valid] / m_tot[valid]

# Save in df
static_model_outputs_df['wC_from_p'] = w_C
static_model_outputs_df['wN_from_p'] = w_N
static_model_outputs_df['wH_from_p'] = w_H
static_model_outputs_df['wO_from_p'] = w_O

# Nx4 array in C,N,H,O order
w_CNHO_from_p = static_model_outputs_df[['wC_from_p', 'wN_from_p', 'wH_from_p', 'wO_from_p']].to_numpy()


In [12]:
# Normalized atomic (molar) fractions from elemental moles (order: C, H, N, O)
n_tot_atoms = n_C + n_H + n_N + n_O
valid_atoms = n_tot_atoms > 0

nC_from_p = np.zeros_like(n_tot_atoms, dtype=float)
nH_from_p = np.zeros_like(n_tot_atoms, dtype=float)
nN_from_p = np.zeros_like(n_tot_atoms, dtype=float)
nO_from_p = np.zeros_like(n_tot_atoms, dtype=float)

nC_from_p[valid_atoms] = n_C[valid_atoms] / n_tot_atoms[valid_atoms]
nH_from_p[valid_atoms] = n_H[valid_atoms] / n_tot_atoms[valid_atoms]
nN_from_p[valid_atoms] = n_N[valid_atoms] / n_tot_atoms[valid_atoms]
nO_from_p[valid_atoms] = n_O[valid_atoms] / n_tot_atoms[valid_atoms]

# Nx4 array in CHNO order
n_CHNO_from_p = np.column_stack((nC_from_p, nH_from_p, nN_from_p, nO_from_p))

# Optional: store in dataframe
static_model_outputs_df["nC_from_p"] = nC_from_p
static_model_outputs_df["nH_from_p"] = nH_from_p
static_model_outputs_df["nN_from_p"] = nN_from_p
static_model_outputs_df["nO_from_p"] = nO_from_p

static_model_outputs_df[["nC_from_p", "nH_from_p", "nN_from_p", "nO_from_p"]].head()

,nC_from_p,nH_from_p,nN_from_p,nO_from_p
0,0.293245,0.093302,0.0,0.613452
1,0.376801,0.017222,0.0,0.605977
2,0.292018,0.095535,0.0,0.612447
3,0.335998,0.042028,0.0,0.621974
4,0.291934,0.093657,0.0,0.614409


In [ ]:
# Conduct batch run with conditional atmospheric inputs
n = min(len(T_eqs), len(T_ints), len(P_surfs), len(m_CNHO_atms), 
        len(Ms), len(Rs), len(static_model_outputs_df))

outs = [None] * n
failed_runs = []

#extract model names from the DataFrame, handling potential variations in column naming
model_col = 'Model' if 'Model' in static_model_outputs_df.columns else 'Model '
model_vals = static_model_outputs_df[model_col].astype(str).str.strip().to_numpy()

for i in range(n):
    try:
        # Base positional args
        M_s = Ms[i]
        R_s = Rs[i]
        T_eq = T_eqs[i]
        T_int = T_ints[i]
        P_s = P_surfs[i]

        # Composition rule: use atomic mass fractions from partial pressures for specific models,
        # otherwise use provided atmospheric masses.
        # This is because the "neongooey" and "moai" models report inconsistent/incomplete elemental
        # masses. So we use the atomic composition from partial pressures as an alternative.
        if model_vals[i] in ("neongooey", "moai"):
            comp_in = w_CNHO_from_p[i]
            print(f"Using w_CNHO_from_p for index {i} due to model '{model_vals[i]}'")
        else:
            comp_in = m_CNHO_atms[i]

        # Check the C/H ratio, if the atmospheric composition is very H-rich (C/H < 0.8),
        # we use the fixed molecular speciation from the partial pressures instead of the default,
        # chemical equilibrium speciation. This is because MOAChi is designed to handle C-rich 
        # atmospheres.

        extra_kwargs = {}
        if comp_in[0]/comp_in[2]/12.<0.8:
        #if (model_vals[i] not in ("neongooey", "moai")) and np.isclose(w_C[i], 0.0):
            print(f"Adding extra_kwarg: ys_atm_raw for index {i} due to low C/H ratio (C/H={comp_in[0]/comp_in[2]:.3f})")
            print(f"also adding O_treatment='fixed_molecular_reservoir' for index {i}")
            extra_kwargs["ys_atm_raw"] = p_gases_ordered[i]
            extra_kwargs["O_treatment"] = "fixed_molecular_reservoir"


        #compute the atmospheric structure and radiative fluxes using MOAChi
        outs[i] = calc_rad_atm_MO_spec_CNH_Teq_fixed_Atm_CNH(
            M_s, R_s,
            T_eq, T_int,
            P_s,
            comp_in,
            initial_guess=[1.0004, 1.07],
            **extra_kwargs
        )

    except Exception as e:
        failed_runs.append((i, str(e)))

print(f"Total runs: {n}")
print(f"Successful: {sum(o is not None for o in outs)}")
print(f"Failed: {len(failed_runs)}")
if failed_runs:
    print("Failed indices:", [i for i, _ in failed_runs])


Iter  8
MR_top	[1.00018285 1.03686748]
rel_err	[2.01832728e-09 3.68440014e-08]
M_s 1.0
M_toa 1.0001828515955578
R_toa 1.0368674412783032
yPs_true [3.04185176e-07 3.84545629e+05 1.22374152e+07 1.24064884e+06
 6.47609790e+01 6.79111900e+06 0.00000000e+00]
M_s_true, R_s_true, CMF 1.000000000002244 1.0000000027073845 0.24590751107418163
M_MO_frac 0.0
Relative error: 
 [2.24398278e-12 2.70738454e-09]
Total runs: 29
Successful: 29
Failed: 0


In [ ]:
#calculate the MO volume fraction from the output object
def find_MO_sol_index(out_obj):
    delta_M = (out_obj.M_s - out_obj.mantle_profiles['M']/M_E)- \
            out_obj.M_MO_frac * out_obj.M_s
    temp1 = delta_M[1:]*delta_M[:-1]
    sol_index = np.where(temp1 < 0)[0][0]
    return sol_index, delta_M

def calc_V_frac_MO(out):
    sol_index, delta_M = find_MO_sol_index(out)
    factor = delta_M[sol_index+1]/(delta_M[sol_index+1]-delta_M[sol_index])
    r_MO = out.mantle_profiles['r'][sol_index + 1] + \
        factor*(out.mantle_profiles['r'][sol_index] - out.mantle_profiles['r'][sol_index + 1])
    r_MO_E = r_MO / R_E
    V_frac_MO = 1. - (r_MO_E/out.R_s_true)**3.
    return V_frac_MO

In [111]:
#save CHILI-style output to CSV given MOAChi output object
def save_CHILI_output_CSV(out, filename):
    #y_O2, y_H2, y_CO2, y_CO, y_CH4, y_H2O
    zs = (out.rs - out.R_s_true)[1:] * R_E
    Ps = out.PTMtaus_a[0][1:]/1e5
    Ts = out.PTMtaus_a[1][1:]
    yss = out.LO[0]
    Pss = Ps[np.newaxis, :] * yss
    #append surface values
    zs[-1] = 0.
    Ps[-1] = out.PTMtau_s[0]/1e5
    Ts[-1] = out.PTMtau_s[1]
    Pss[:,-1] = out.yPs_surf/1e5
    
    #reorder Pss gas species into H2O, CO2, CO, H2, CH4, O2
    Pss_reordered = np.zeros_like(Pss[1:])

    Pss_reordered[0,:] = Pss[5,:]  #H2O
    Pss_reordered[1,:] = Pss[2,:]  #CO2
    Pss_reordered[2,:] = Pss[3,:]  #CO
    Pss_reordered[3,:] = Pss[1,:]  #H2
    Pss_reordered[4,:] = Pss[4,:]  #CH4
    Pss_reordered[5,:] = Pss[0,:]  #O2
    data_to_save = np.concatenate((zs[np.newaxis, :], Ps[np.newaxis, :],
                                    Ts[np.newaxis, :], Pss_reordered), axis=0)
    header_str = 'z(m),p_tot(bar),T(K),p_H2O(bar),p_CO2(bar),p_CO(bar),p_H2(bar),p_CH4(bar),p_O2(bar)'
    np.savetxt(filename, data_to_save.T, delimiter=',', header=header_str)
    print(f'Saved CHILI-style output to {filename}')


In [13]:
# Build per-row output filenames in the format:
# static-moachi-planet-tau<number>-<case>-data.csv
# get rid of whitespace and lowercase the planet and case names as well as hyphens 


planet_col = next((c for c in static_model_outputs_df.columns if c.strip().lower() == "planet"), None)
case_col = next((c for c in static_model_outputs_df.columns if c.strip().lower() == "case"), None)
tau_col = next((c for c in static_model_outputs_df.columns if c.strip() in ("τ", "tau")), None)

if not all([planet_col, case_col, tau_col]):
    raise KeyError(f"Could not find required columns. Columns are: {list(static_model_outputs_df.columns)}")

def _clean_text(x):
    return str(x).strip().lower().replace("-", "")

def _tau_fmt(x):
    return f"{float(x):g}"  # 3.0 -> 3, 3.5 -> 3.5

static_model_outputs_df["output_filename"] = static_model_outputs_df.apply(
    lambda r: f"static-moachi-{_clean_text(r[planet_col])}-tau{_tau_fmt(r[tau_col])}-{_clean_text(r[case_col])}-data.csv",
    axis=1
)

file_names = static_model_outputs_df["output_filename"].tolist()
static_model_outputs_df[[planet_col, tau_col, case_col, "output_filename"]].head()

,planet,τ,Case,output_filename
0,trappist-1b,3.0,Hot,static-moachi-trappist1b-tau3-hot-data.csv
1,trappist-1b,3.0,Cold,static-moachi-trappist1b-tau3-cold-data.csv
2,trappist-1b,4.0,Hot,static-moachi-trappist1b-tau4-hot-data.csv
3,trappist-1b,4.0,Cold,static-moachi-trappist1b-tau4-cold-data.csv
4,trappist-1b,6.0,Hot,static-moachi-trappist1b-tau6-hot-data.csv


In [ ]:
#save CHILI-style output files for each row in the DataFrame, using the same logic as used for the batch run.

from pathlib import Path

out_dir = Path("CHILI_out")
out_dir.mkdir(parents=True, exist_ok=True)

if "output_filename" not in static_model_outputs_df.columns:
    raise KeyError("Column 'output_filename' not found in static_model_outputs_df.")

if "outs" not in globals():
    raise NameError("Variable 'outs' not found. Run the batch solver cell first.")

n = min(len(static_model_outputs_df), len(outs))
saved, skipped, failed = 0, [], []

for i in range(n):
    out_i = outs[i]
    if out_i is None:
        skipped.append(i)
        continue

    fpath = out_dir / static_model_outputs_df.loc[i, "output_filename"]
    try:
        save_CHILI_output_CSV(out_i, str(fpath))
        saved += 1
    except Exception as e:
        failed.append((i, str(e)))

print(f"Requested rows: {len(static_model_outputs_df)}")
print(f"Processed rows: {n}")
print(f"Saved files: {saved}")
print(f"Skipped (out is None): {len(skipped)}")
if skipped:
    print("Skipped indices:", skipped)
print(f"Failed saves: {len(failed)}")
if failed:
    print("Failed indices:", [i for i, _ in failed])

Saved CHILI-style output to CHILI_out/static-moachi-trappist1b-tau3-hot-data.csv
Saved CHILI-style output to CHILI_out/static-moachi-trappist1b-tau3-cold-data.csv
Saved CHILI-style output to CHILI_out/static-moachi-trappist1b-tau4-hot-data.csv
Saved CHILI-style output to CHILI_out/static-moachi-trappist1b-tau4-cold-data.csv
Saved CHILI-style output to CHILI_out/static-moachi-trappist1b-tau6-hot-data.csv
Saved CHILI-style output to CHILI_out/static-moachi-trappist1b-tau6-cold-data.csv
Saved CHILI-style output to CHILI_out/static-moachi-trappist1alpha-tau3-hot-data.csv
Saved CHILI-style output to CHILI_out/static-moachi-trappist1alpha-tau3-cold-data.csv
Saved CHILI-style output to CHILI_out/static-moachi-trappist1alpha-tau4-hot-data.csv
Saved CHILI-style output to CHILI_out/static-moachi-trappist1alpha-tau4-cold-data.csv
Saved CHILI-style output to CHILI_out/static-moachi-trappist1alpha-tau6-hot-data.csv
Saved CHILI-style output to CHILI_out/static-moachi-trappist1e-tau3-hot-data.csv
Sav

In [14]:
# Generate config filenames: static-moachi-planet-tau<number>-<case>-config.in

planet_col = next((c for c in static_model_outputs_df.columns if c.strip().lower() == "planet"), None)
case_col = next((c for c in static_model_outputs_df.columns if c.strip().lower() == "case"), None)
tau_col = next((c for c in static_model_outputs_df.columns if c.strip() in ("τ", "tau")), None)

if not all([planet_col, case_col, tau_col]):
    raise KeyError(f"Could not find required columns. Columns are: {list(static_model_outputs_df.columns)}")

def _clean_text(x):
    return str(x).strip().lower().replace("-", "").replace(" ", "")

def _tau_fmt(x):
    return f"{float(x):g}"

static_model_outputs_df["config_filename"] = static_model_outputs_df.apply(
    lambda r: f"static-moachi-{_clean_text(r[planet_col])}-tau{_tau_fmt(r[tau_col])}-{_clean_text(r[case_col])}-config.in",
    axis=1
)

config_file_names = static_model_outputs_df["config_filename"].tolist()
static_model_outputs_df[[planet_col, tau_col, case_col, "config_filename"]].head()

,planet,τ,Case,config_filename
0,trappist-1b,3.0,Hot,static-moachi-trappist1b-tau3-hot-config.in
1,trappist-1b,3.0,Cold,static-moachi-trappist1b-tau3-cold-config.in
2,trappist-1b,4.0,Hot,static-moachi-trappist1b-tau4-hot-config.in
3,trappist-1b,4.0,Cold,static-moachi-trappist1b-tau4-cold-config.in
4,trappist-1b,6.0,Hot,static-moachi-trappist1b-tau6-hot-config.in


In [ ]:
#generate CHILI-style config files for each row in the DataFrame, 
# using the same logic as used for the batch run.

from pathlib import Path
import json

# output folder for configs
cfg_dir = Path("configs")
cfg_dir.mkdir(parents=True, exist_ok=True)

if "config_filename" not in static_model_outputs_df.columns:
    raise KeyError("Column 'config_filename' not found in static_model_outputs_df.")

# determine model column name robustly (as used elsewhere)
model_col = 'Model' if 'Model' in static_model_outputs_df.columns else 'Model '
model_vals = static_model_outputs_df[model_col].astype(str).str.strip().to_numpy()

n = len(static_model_outputs_df)

saved_files = []
skipped = []
errors = []

for i in range(n):
    try:
        # positional inputs (recompute same way as used for runs)
        M_s = float(Ms[i])
        R_s = float(Rs[i])
        T_eq = float(T_eqs[i])
        T_int = float(T_ints[i])
        P_s = float(P_surfs[i])

        # composition selection follows previous rule
        if model_vals[i] in ("neongooey", "moai"):
            comp_in = w_CNHO_from_p[i].astype(float)
            comp_type = "surface_p_i"
        else:
            comp_in = m_CNHO_atms[i].astype(float)
            comp_type = "bulk_mass_atm"

        # decide whether ys_atm_raw / O_treatment would have been added
        extra_kwargs = {}
        # guard division by zero
        if comp_in[2] != 0 and (comp_in[0] / comp_in[2] / 12.0) < 0.8:
            extra_kwargs["ys_atm_raw"] = p_gases_ordered[i].tolist()
            extra_kwargs["O_treatment"] = "fixed_molecular_reservoir"

        b_ChemEq = "ys_atm_raw" not in extra_kwargs

        # build config dict with all actual parameters used
        cfg = {
            "idx": int(i),
            "M_s": float(M_s),
            "R_s": float(R_s),
            "T_eq": float(T_eq),
            "T_int": float(T_int),
            "P_s_Pa": float(P_s),  # P_s already in Pa in notebook
            "elemental_comp_source": comp_type,
            "C_N_H_O_mass_frac": comp_in.tolist(),
            "initial_guess": [1.0004, 1.07],
            "b_ChemEq": bool(b_ChemEq),
        }
        # add extra kwargs if present
        if "ys_atm_raw" in extra_kwargs:
            cfg["ys_atm_raw"] = extra_kwargs["ys_atm_raw"]
            #cfg["O_treatment"] = extra_kwargs["O_treatment"]

        # write config file as simple IN-like format (key = json)
        fname = static_model_outputs_df.loc[i, "config_filename"]
        fpath = cfg_dir / fname

        with open(fpath, "w") as f:
            f.write("# auto-generated config.in for row {}\n".format(i))
            for k, v in cfg.items():
                # store arrays/lists as JSON strings for reproducibility
                if isinstance(v, (list, tuple, np.ndarray, dict)):
                    f.write(f"{k} = {json.dumps(v)}\n")
                else:
                    f.write(f"{k} = {repr(v)}\n")

        # print the saved content for user inspection
        print(f"Saved config: {fpath}")
        with open(fpath, "r") as f:
            print(f.read())

        saved_files.append(str(fpath))

    except Exception as e:
        errors.append((i, str(e)))
        skipped.append(i)
        continue

print(f"Total configs requested: {n}")
print(f"Saved configs: {len(saved_files)}")
if errors:
    print(f"Errors for indices: {[i for i, _ in errors]}")

Saved config: configs/static-moachi-trappist1b-tau3-hot-config.in
# auto-generated config.in for row 0
idx = 0
M_s = 1.0
R_s = 1.0
T_eq = 1320.6404864438753
T_int = 827.5989082466499
P_s_Pa = 17254030.53118
elemental_comp_source = 'surface_p_i'
C_N_H_O_mass_frac = [0.26207033632723953, 0.0, 0.0069486064511211434, 0.7309810572216393]
initial_guess = [1.0004, 1.07]
b_ChemEq = True

Saved config: configs/static-moachi-trappist1b-tau3-cold-config.in
# auto-generated config.in for row 1
idx = 1
M_s = 1.0
R_s = 1.0
T_eq = 615.9826504912061
T_int = 1104.4917151364807
P_s_Pa = 13855144.476864578
elemental_comp_source = 'bulk_mass_atm'
C_N_H_O_mass_frac = [0.3176525385135197, 0.0, 0.0012098753948837635, 0.6811375860915966]
initial_guess = [1.0004, 1.07]
b_ChemEq = True

Saved config: configs/static-moachi-trappist1b-tau4-hot-config.in
# auto-generated config.in for row 2
idx = 2
M_s = 1.0
R_s = 1.0
T_eq = 1320.6404864438753
T_int = 130.09255776753878
P_s_Pa = 17519464.588150002
elemental_comp_s